# 08 - PyTorch Bridge

这一节把 micrograd 的概念翻译到 PyTorch。

这一节只抓一句话：

> micrograd 是标量版自动求导；PyTorch 是张量版自动求导。

这一节开始，**PyTorch 是必装依赖**，不是可选阅读。你前面学的 `data / grad / backward / zero_grad / update`，在 PyTorch 里都要亲手跑一遍。

<!-- xiao-preview:start -->
## 课前预习作业：先把词对上

先不写 PyTorch。把 micrograd 里最熟的三个动作翻译一下。

In [ ]:
# TODO: 把 None 改成你的答案。
# L = w^2, w = 3
preview_grad = None                 # dL/dw 是多少？
preview_backward_name = None        # 触发反向传播的方法名，填字符串，比如 'xxx'
preview_need_zero_grad = None       # 每轮训练前是否要清空旧梯度？填 True 或 False


def qa_check_08_preview(grad, backward_name, need_zero_grad):
    if grad is None or backward_name is None or need_zero_grad is None:
        print('请先填写 TODO：先把 micrograd 的动作对上。')
        return
    assert grad == 6, f'd(w^2)/dw at w=3 应该是 6，但你填的是 {grad}'
    assert backward_name == 'backward', "反向传播方法名是 'backward'。"
    assert need_zero_grad is True, '训练循环里需要清空旧梯度。'
    print('OK: PyTorch 对照预习通过。')


qa_check_08_preview(preview_grad, preview_backward_name, preview_need_zero_grad)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

`L = w^2`，导数是 `2w`。micrograd 和 PyTorch 都用 `.backward()` 触发反向传播。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_grad = 6
preview_backward_name = 'backward'
preview_need_zero_grad = True
```

</details>

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import random

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
from micrograd.nn import MLP

try:
    import torch
except ModuleNotFoundError as exc:
    install_message = (
        '08_pytorch_bridge 开始 PyTorch 是必装依赖。请安装到 notebook 使用的 .venv，不要装到系统 Python:\n'
        '  /Users/barry/IdeaProjects/llm/.venv/bin/python -m pip install torch\n'
        '如果你使用 conda/uv 或 GPU 环境，请按 https://pytorch.org/get-started/locally/ 选择对应命令，但仍要装进当前 notebook kernel 对应环境。'
    )
    raise ModuleNotFoundError(install_message) from exc

random.seed(1337)
print('project root:', PROJECT_ROOT)
print('torch version:', torch.__version__)

## 1. 概念对照表

| micrograd | PyTorch | 先这样理解 |
|---|---|---|
| `Value(2.0)` | `torch.tensor(2.0, requires_grad=True)` | 一个会记录梯度的数 |
| `.data` | `.item()` 或 tensor 本身 | 前向算出来的值 |
| `.grad` | `.grad` | 最终 loss 对它的导数 |
| `loss.backward()` | `loss.backward()` | 从 loss 反向传播 |
| `model.zero_grad()` | `optimizer.zero_grad()` 或手动清零 | 清空旧梯度 |
| 手写更新 `p.data += ...` | `optimizer.step()` | 用梯度更新参数 |

关键差别：micrograd 主要是一颗颗标量 `Value`；PyTorch 的 tensor 可以一次装很多数。

## 2. micrograd 版本：一个参数的一步训练

目标：让 `w * x` 接近 `y`。

In [ ]:
w = Value(2.0)
x = Value(3.0)
y = 7.0
learning_rate = 0.1

pred = w * x
loss = (pred - y) ** 2

print('pred:', pred.data)
print('loss:', loss.data)

loss.backward()
print('w.grad:', w.grad)

w.data += -learning_rate * w.grad
print('w after update:', w.data)

## 3. PyTorch 版本：同一件事

下面这段代码必须实际运行。进入这一节后，PyTorch 不再是可选项；它是把 micrograd 概念迁移到真实深度学习框架的硬门槛。

In [ ]:
w_t = torch.tensor(2.0, requires_grad=True)
x_t = torch.tensor(3.0)
y_t = torch.tensor(7.0)
learning_rate = 0.1

pred_t = w_t * x_t
loss_t = (pred_t - y_t) ** 2

print('pred:', pred_t.item())
print('loss:', loss_t.item())

loss_t.backward()
print('w.grad:', w_t.grad.item())

with torch.no_grad():
    w_t += -learning_rate * w_t.grad

print('w after update:', w_t.item())

## 4. 为什么 PyTorch 更新时常见 `no_grad`

在 micrograd 里你直接改：

```python
p.data += -lr * p.grad
```

在 PyTorch 里，参数更新本身不应该被记录进下一张计算图，所以常见写法是：

```python
with torch.no_grad():
    p += -lr * p.grad
```

更常见的是让优化器做：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

## 5. TODO Lab - 写 micrograd 的一步更新

TODO：写一个函数，返回更新后的 `w.data`。

给定：

```text
w = 2, x = 3, y = 7, lr = 0.1
pred = w*x = 6
loss = (pred-y)^2 = 1
w.grad = 2*(pred-y)*x = -6
w_new = 2 - 0.1*(-6) = 2.6
```

In [ ]:
def one_step_micrograd(w_data, x_data, y_target, learning_rate):
    # TODO: 用 Value 完成 pred、loss、backward、update，最后返回 w.data
    pass


def qa_check_08_one_step(func):
    out = func(2.0, 3.0, 7.0, 0.1)
    if out is None:
        print('请先填写 TODO：返回更新后的 w.data。')
        return
    assert abs(out - 2.6) < 1e-9, f'expected 2.6, got {out}'
    print('OK: one_step_micrograd passed.')


qa_check_08_one_step(one_step_micrograd)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

函数里只需要一个参数 `w = Value(w_data)`。`x` 可以是 `Value(x_data)`，也可以直接用普通数字，因为 micrograd 会包装普通数字。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def one_step_micrograd(w_data, x_data, y_target, learning_rate):
    w = Value(w_data)
    x = Value(x_data)
    pred = w * x
    loss = (pred - y_target) ** 2
    loss.backward()
    w.data += -learning_rate * w.grad
    return w.data
```

</details>

## What To Remember

```text
1. 08 开始 PyTorch 是必装依赖，不再跳过。
2. micrograd 的 Value 可以看成 PyTorch Tensor 的标量玩具版。
3. 两边都有 data/value、grad、backward、zero_grad、update。
4. PyTorch Tensor 可以装向量、矩阵、批量数据。
5. PyTorch 更新参数时通常用 optimizer.step()。
6. 你理解 micrograd，PyTorch 的自动求导就不再是黑盒。
```

<!-- xiao-post:start -->
## 课后作业提交清单

```text
1. 我已经在当前环境安装并 import 过 PyTorch。
2. 我能把 Value 对应到 torch.tensor(..., requires_grad=True)。
3. 我能解释 PyTorch 里的 .grad 仍然是 loss 对参数的导数。
4. 我能说明 optimizer.zero_grad() / loss.backward() / optimizer.step() 的顺序。
5. 我跑过 micrograd 和 PyTorch 两个 one_step，并看到 w 都从 2.0 变成 2.6。
```

## AI 复盘检查 Prompt

```text
你是我的 micrograd 到 PyTorch 迁移检查官。
请你一次只问一个问题，检查我是否理解：
1. Value 和 torch.tensor(requires_grad=True) 的关系
2. micrograd 的 grad 和 PyTorch 的 .grad 是否同义
3. 为什么 PyTorch 更新参数时要 zero_grad/backward/step
4. 为什么更新参数通常不记录进计算图
5. micrograd 标量自动求导和 PyTorch 张量自动求导的最大区别
如果我答错，请用 w=2, x=3, y=7 的一步训练例子提示我。
```